## Co-location Table 1: ERA5 Bilinearly Interpolated at DPC Station Coordinates

For each DPC station, the four nearest ERA5 grid points surrounding its coordinates are identified, and the corresponding gridded ERA5 value is bilinearly interpolated to the station's exact latitude and longitude.

## 0. Imports and Configuration

In [2]:
import numpy as np
import pandas as pd
import xarray as xr
import os
import json
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──
PROJECT_ROOT = r'D:\StormEngine'

DPC_STATIONS_PATH = os.path.join(PROJECT_ROOT, 'DataAggregation', 'DPC',
                                  'DPC_stations_aggregated_dq.csv')

ERA5_DIR = os.path.join(PROJECT_ROOT, 'Historical_data_era5', 'estratto_std')
ERA5_FILES = [
    os.path.join(ERA5_DIR, f'era5_std_adriatico_2024_{month:02d}.nc')
    for month in range(1, 13)
]

OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'DataAggregation', 'Colocation')
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'colocation_era5_dpc.csv')

# ERA5 standard variables and their unit conversions (raw NetCDF -> standard units)
ERA5_VARS = {
    'msl'  : {'unit': 'hPa', 'convert': lambda x: x / 100.0},   # Pa -> hPa
    'u10'  : {'unit': 'm/s', 'convert': lambda x: x},
    'v10'  : {'unit': 'm/s', 'convert': lambda x: x},
    'i10fg': {'unit': 'm/s', 'convert': lambda x: x},
}

# Domain bounds (sanity check only)
LAT_MIN, LAT_MAX = 39.0, 46.5
LON_MIN, LON_MAX = 12.0, 20.0

print('Configuration loaded.')
print(f'DPC stations file : {DPC_STATIONS_PATH}')
print(f'ERA5 files dir     : {ERA5_DIR}')
print(f'Output             : {OUTPUT_CSV}')

Configuration loaded.
DPC stations file : D:\StormEngine\DataAggregation\DPC\DPC_stations_aggregated_dq.csv
ERA5 files dir     : D:\StormEngine\Historical_data_era5\estratto_std
Output             : D:\StormEngine\DataAggregation\Colocation\colocation_era5_dpc.csv


## 1. Load DPC Stations: Automatic Schema Detection

This cell auto-detects which shape the file actually has and always reduces the result to one row per unique station with `(station_id, station_name, lat, lon)`: the only information Co-location Table 1 actually needs from the DPC side.

In [3]:
def load_dpc_stations(path):
    """
    Loads the DPC stations file and returns a DataFrame with exactly one
    row per unique station: columns station_id, station_name, lat, lon.
    Auto-detects wide (xxx_value columns) vs long (sensor_code/value) format.
    """
    df = pd.read_csv(path)
    print(f'Raw file shape : {df.shape}')
    print(f'Columns found  : {list(df.columns)}')

    value_cols = [c for c in df.columns if c.endswith('_value')]
    is_wide = len(value_cols) > 0
    is_long = ('sensor_code' in df.columns) and ('value' in df.columns)

    if is_wide:
        fmt = 'wide'
    elif is_long:
        fmt = 'long'
    else:
        raise ValueError(
            'Could not detect DPC file format: no *_value columns found '
            "and no ('sensor_code','value') pair found. Inspect columns manually."
        )

    print(f'\nDetected format: {fmt.upper()}')

    # Identify the station_id, station_name, lat, lon columns by best-effort matching
    def find_col(candidates):
        for c in candidates:
            if c in df.columns:
                return c
        return None

    id_col   = find_col(['station_id', 'id'])
    name_col = find_col(['station_name', 'sensor_name', 'name'])
    lat_col  = find_col(['lat', 'latitude'])
    lon_col  = find_col(['lon', 'longitude'])

    print(f'station_id  -> {id_col}')
    print(f'station_name -> {name_col}')
    print(f'lat         -> {lat_col}')
    print(f'lon         -> {lon_col}')

    assert id_col and lat_col and lon_col, 'Could not identify required columns.'

    keep_cols = [id_col, lat_col, lon_col] + ([name_col] if name_col else [])
    stations = df[keep_cols].drop_duplicates(subset=[id_col]).copy()
    stations = stations.rename(columns={
        id_col: 'station_id', lat_col: 'lat', lon_col: 'lon',
        **({name_col: 'station_name'} if name_col else {})
    })
    if 'station_name' not in stations.columns:
        stations['station_name'] = stations['station_id'].astype(str)

    stations['lat'] = pd.to_numeric(stations['lat'], errors='coerce')
    stations['lon'] = pd.to_numeric(stations['lon'], errors='coerce')
    stations = stations.dropna(subset=['lat', 'lon']).reset_index(drop=True)

    return stations, fmt


dpc_stations, dpc_format = load_dpc_stations(DPC_STATIONS_PATH)

print(f'\nUnique DPC stations with valid coordinates: {len(dpc_stations)}')
print(dpc_stations.head(10).to_string(index=False))

Raw file shape : (752, 34)
Columns found  : ['station_id', 'station_name', 'lat', 'lon', 'gestore', 'source', 'PREC_value', 'PREC_dt', 'TARIA2M_value', 'TARIA2M_dt', 'PRESS_value', 'PRESS_dt', 'VV_value', 'VV_dt', 'UMID2M_value', 'UMID2M_dt', 'PREC_range_flag', 'TARIA2M_range_flag', 'PRESS_range_flag', 'VV_range_flag', 'UMID2M_range_flag', 'PREC_robust_z', 'PREC_outlier_flag', 'TARIA2M_robust_z', 'TARIA2M_outlier_flag', 'PRESS_robust_z', 'PRESS_outlier_flag', 'VV_robust_z', 'VV_outlier_flag', 'UMID2M_robust_z', 'UMID2M_outlier_flag', 'any_outlier_flag', 'dist_to_coast_km', 'coastal_weight']

Detected format: WIDE
station_id  -> station_id
station_name -> station_name
lat         -> lat
lon         -> lon

Unique DPC stations with valid coordinates: 752
  station_id      lat      lon       station_name
 METEOHUB::1 43.62819 12.68448         Acqualagna
 METEOHUB::2 43.54330 13.38060          Agugliano
 METEOHUB::3 42.97638 13.35133           Amandola
 METEOHUB::4 43.61030 13.50830     An

## 1.1 Domain Sanity Check

Confirms that DPC station coordinates fall within (or close to) the ERA5 domain used throughout the project, before attempting interpolation.

In [4]:
in_domain = (
    (dpc_stations['lat'] >= LAT_MIN) & (dpc_stations['lat'] <= LAT_MAX) &
    (dpc_stations['lon'] >= LON_MIN) & (dpc_stations['lon'] <= LON_MAX)
)

print(f'Stations inside ERA5 domain [{LAT_MIN}-{LAT_MAX}°N, {LON_MIN}-{LON_MAX}°E]: '
      f'{in_domain.sum()} / {len(dpc_stations)}')

if (~in_domain).sum() > 0:
    print('\nStations OUTSIDE the domain (will be clipped to nearest edge during interpolation):')
    print(dpc_stations[~in_domain][['station_id', 'station_name', 'lat', 'lon']].to_string(index=False))

Stations inside ERA5 domain [39.0-46.5°N, 12.0-20.0°E]: 469 / 752

Stations OUTSIDE the domain (will be clipped to nearest edge during interpolation):
station_id                          station_name       lat       lon
  ARPAE::4     ARPAE Station STA_43.8318_11.8377 43.831750 11.837710
  ARPAE::6     ARPAE Station STA_43.8701_11.8387 43.870080 11.838660
  ARPAE::7     ARPAE Station STA_43.8718_11.7472 43.871780 11.747200
 ARPAE::12     ARPAE Station STA_43.9031_11.7992 43.903150 11.799160
 ARPAE::13     ARPAE Station STA_43.9036_11.9011 43.903580 11.901140
 ARPAE::14     ARPAE Station STA_43.9071_11.7931 43.907080 11.793140
 ARPAE::15     ARPAE Station STA_43.9226_11.9577 43.922640 11.957660
 ARPAE::16     ARPAE Station STA_43.9243_11.7933 43.924260 11.793300
 ARPAE::17     ARPAE Station STA_43.9252_11.8919 43.925250 11.891860
 ARPAE::27     ARPAE Station STA_43.9935_11.9453 43.993490 11.945300
 ARPAE::30     ARPAE Station STA_44.0024_11.6654 44.002370 11.665360
 ARPAE::31     ARPAE 

## 2. Load and Concatenate the 12 Monthly ERA5 Files

Same procedure used throughout the seasonal analysis: load all 12 monthly NetCDF files and concatenate them along the time dimension into one continuous yearly dataset.

In [5]:
missing = [f for f in ERA5_FILES if not os.path.isfile(f)]
print(f'Expected {len(ERA5_FILES)} ERA5 monthly files:')
for f in ERA5_FILES:
    print(f'  {"OK" if os.path.isfile(f) else "MISSING":8s} {f}')

assert not missing, f'Missing {len(missing)} ERA5 file(s): {missing}'

era5_datasets = [xr.open_dataset(f) for f in ERA5_FILES]
era5_year = xr.concat(era5_datasets, dim='valid_time').sortby('valid_time')

era5_lats = era5_year['latitude'].values   # descending, e.g. 46.5 ... 39.0
era5_lons = era5_year['longitude'].values  # ascending,  e.g. 12.0 ... 20.0
era5_times = pd.DatetimeIndex(era5_year['valid_time'].values)

print(f'\nERA5 yearly dataset:')
print(f'  Total hours : {era5_year.sizes["valid_time"]}  (expected 8784 for 2024)')
print(f'  Grid        : {era5_year.sizes["latitude"]} x {era5_year.sizes["longitude"]}')
print(f'  Variables   : {list(era5_year.data_vars)}')
print(f'  Lat range   : {era5_lats.min():.2f} to {era5_lats.max():.2f}')
print(f'  Lon range   : {era5_lons.min():.2f} to {era5_lons.max():.2f}')

Expected 12 ERA5 monthly files:
  OK       D:\StormEngine\Historical_data_era5\estratto_std\era5_std_adriatico_2024_01.nc
  OK       D:\StormEngine\Historical_data_era5\estratto_std\era5_std_adriatico_2024_02.nc
  OK       D:\StormEngine\Historical_data_era5\estratto_std\era5_std_adriatico_2024_03.nc
  OK       D:\StormEngine\Historical_data_era5\estratto_std\era5_std_adriatico_2024_04.nc
  OK       D:\StormEngine\Historical_data_era5\estratto_std\era5_std_adriatico_2024_05.nc
  OK       D:\StormEngine\Historical_data_era5\estratto_std\era5_std_adriatico_2024_06.nc
  OK       D:\StormEngine\Historical_data_era5\estratto_std\era5_std_adriatico_2024_07.nc
  OK       D:\StormEngine\Historical_data_era5\estratto_std\era5_std_adriatico_2024_08.nc
  OK       D:\StormEngine\Historical_data_era5\estratto_std\era5_std_adriatico_2024_09.nc
  OK       D:\StormEngine\Historical_data_era5\estratto_std\era5_std_adriatico_2024_10.nc
  OK       D:\StormEngine\Historical_data_era5\estratto_std\era5_std

## 3. Identify the Four Surrounding ERA5 Grid Points per DPC Station

For each station, this finds the indices of the grid cell that brackets its (lat, lon): the two surrounding latitudes and two surrounding longitudes, used by the bilinear interpolation in the next step. Coordinates falling outside the grid are clipped to the domain edge (this matches the warning already raised in the domain sanity check above).

In [6]:
def find_bracketing_indices(coord, grid):
    """
    Given a 1D coordinate array `grid` (ascending or descending) and a query
    value `coord`, returns (i0, i1, frac) where grid[i0] and grid[i1] are the
    two bracketing grid points and frac in [0,1] is the interpolation weight
    toward grid[i1]. Clips queries outside the grid to the nearest edge.
    """
    ascending = grid[0] < grid[-1]
    g = grid if ascending else grid[::-1]

    coord_clipped = np.clip(coord, g[0], g[-1])
    idx = np.searchsorted(g, coord_clipped)
    idx = np.clip(idx, 1, len(g) - 1)

    i1 = idx
    i0 = idx - 1
    span = g[i1] - g[i0]
    frac = 0.0 if span == 0 else (coord_clipped - g[i0]) / span

    if not ascending:
        i0, i1 = len(grid) - 1 - i0, len(grid) - 1 - i1

    return int(i0), int(i1), float(frac)


bracket_info = []
for _, row in dpc_stations.iterrows():
    lat_i0, lat_i1, lat_frac = find_bracketing_indices(row['lat'], era5_lats)
    lon_i0, lon_i1, lon_frac = find_bracketing_indices(row['lon'], era5_lons)
    bracket_info.append({
        'station_id': row['station_id'],
        'lat_idx_0': lat_i0, 'lat_idx_1': lat_i1, 'lat_frac': lat_frac,
        'lon_idx_0': lon_i0, 'lon_idx_1': lon_i1, 'lon_frac': lon_frac,
        'era5_lat_0': era5_lats[lat_i0], 'era5_lat_1': era5_lats[lat_i1],
        'era5_lon_0': era5_lons[lon_i0], 'era5_lon_1': era5_lons[lon_i1],
    })

bracket_df = pd.DataFrame(bracket_info)
print(f'Bracketing grid points identified for {len(bracket_df)} stations.')
print('\nSample (first 5 stations):')
print(bracket_df.head(5).to_string(index=False))

Bracketing grid points identified for 752 stations.

Sample (first 5 stations):
 station_id  lat_idx_0  lat_idx_1  lat_frac  lon_idx_0  lon_idx_1  lon_frac  era5_lat_0  era5_lat_1  era5_lon_0  era5_lon_1
METEOHUB::1         12         11   0.51276          2          3   0.73792       43.50       43.75       12.50       12.75
METEOHUB::2         12         11   0.17320          5          6   0.52240       43.50       43.75       13.25       13.50
METEOHUB::3         15         14   0.90552          5          6   0.40532       42.75       43.00       13.25       13.50
METEOHUB::4         12         11   0.44120          6          7   0.03320       43.50       43.75       13.50       13.75
METEOHUB::5         12         11   0.44380          5          6   0.79920       43.50       43.75       13.25       13.50


## 4. Bilinear Interpolation: All Variables, All Timesteps

For each station and each hourly timestep, the four surrounding ERA5 grid values are combined using bilinear interpolation weights derived from the station's exact position within the grid cell.

In [7]:
def bilinear_interpolate_series(data_array, lat_i0, lat_i1, lat_frac,
                                  lon_i0, lon_i1, lon_frac):
    """
    Bilinearly interpolates a (time, lat, lon) array at a fixed fractional
    position within one grid cell, for all timesteps at once.
    Returns a 1D array of length n_time.
    """
    v00 = data_array[:, lat_i0, lon_i0]
    v01 = data_array[:, lat_i0, lon_i1]
    v10 = data_array[:, lat_i1, lon_i0]
    v11 = data_array[:, lat_i1, lon_i1]

    top    = v00 * (1 - lon_frac) + v01 * lon_frac
    bottom = v10 * (1 - lon_frac) + v11 * lon_frac
    return top * (1 - lat_frac) + bottom * lat_frac


# Pre-load raw arrays once per variable (avoids repeated xarray indexing overhead)
raw_arrays = {var: era5_year[var].values for var in ERA5_VARS}
print('Loaded raw ERA5 arrays for:', list(raw_arrays.keys()))

colocation_rows = []

for _, b in bracket_df.iterrows():
    station_id = b['station_id']

    interpolated = {}
    for var, cfg in ERA5_VARS.items():
        ts = bilinear_interpolate_series(
            raw_arrays[var],
            int(b['lat_idx_0']), int(b['lat_idx_1']), b['lat_frac'],
            int(b['lon_idx_0']), int(b['lon_idx_1']), b['lon_frac'],
        )
        interpolated[var] = cfg['convert'](ts)

    for t_idx, t in enumerate(era5_times):
        row = {
            'station_id': station_id,
            'dt': t,
            'lat_idx_0': int(b['lat_idx_0']), 'lat_idx_1': int(b['lat_idx_1']),
            'lon_idx_0': int(b['lon_idx_0']), 'lon_idx_1': int(b['lon_idx_1']),
        }
        for var in ERA5_VARS:
            row[f'era5_{var}'] = float(interpolated[var][t_idx])
        colocation_rows.append(row)

print(f'\nBuilt {len(colocation_rows)} co-location rows '
      f'({len(bracket_df)} stations x {len(era5_times)} hourly timesteps).')

Loaded raw ERA5 arrays for: ['msl', 'u10', 'v10', 'i10fg']

Built 6605568 co-location rows (752 stations x 8784 hourly timesteps).


## 5. Assemble the Final Co-location Table

Joins the interpolated ERA5 values back to the DPC station metadata (name, exact coordinates) and to the four surrounding grid point coordinates, for full traceability of how each value was produced.

In [8]:
colocation_df = pd.DataFrame(colocation_rows)

colocation_df = colocation_df.merge(
    dpc_stations[['station_id', 'station_name', 'lat', 'lon']],
    on='station_id', how='left'
).rename(columns={'lat': 'station_lat', 'lon': 'station_lon'})

colocation_df = colocation_df.merge(
    bracket_df[['station_id', 'era5_lat_0', 'era5_lat_1', 'era5_lon_0', 'era5_lon_1']],
    on='station_id', how='left'
)

preferred_order = [
    'station_id', 'station_name', 'station_lat', 'station_lon', 'dt',
    'era5_msl', 'era5_u10', 'era5_v10', 'era5_i10fg',
    'lat_idx_0', 'lat_idx_1', 'lon_idx_0', 'lon_idx_1',
    'era5_lat_0', 'era5_lat_1', 'era5_lon_0', 'era5_lon_1',
]
colocation_df = colocation_df[[c for c in preferred_order if c in colocation_df.columns]]

print(f'Final co-location table shape: {colocation_df.shape}')
print(f'Unique stations: {colocation_df["station_id"].nunique()}')
print(f'Time range: {colocation_df["dt"].min()} -> {colocation_df["dt"].max()}')
print()
colocation_df.sample(5)

Final co-location table shape: (6605568, 17)
Unique stations: 752
Time range: 2024-01-01 00:00:00 -> 2024-12-31 23:00:00



,station_id,station_name,station_lat,station_lon,dt,era5_msl,era5_u10,era5_v10,era5_i10fg,lat_idx_0,lat_idx_1,lon_idx_0,lon_idx_1,era5_lat_0,era5_lat_1,era5_lon_0,era5_lon_1
3938764,ARPAV::59,Domegge di Cadore,46.460950,12.410360,2024-05-27 04:00:00,1021.278687,0.543543,-0.685640,3.904691,1,0,1,2,46.25,46.50,12.25,12.50
4598633,DPC::135,Pramollo,46.529829,13.303484,2024-07-10 17:00:00,1016.542236,0.027293,0.862960,2.803831,1,0,5,6,46.25,46.50,13.25,13.50
3417509,ARPAV::25,Sospirolo,46.140636,12.075395,2024-01-23 05:00:00,1025.196899,0.802316,-1.141389,7.808068,2,1,0,1,46.00,46.25,12.00,12.25
4982811,DPC::193,Uccea,46.306483,13.400202,2024-04-05 03:00:00,1021.959045,-0.195448,-0.465291,2.469209,1,0,5,6,46.25,46.50,13.25,13.50
5453814,DPC::40,Col della Gallina meteo,46.055336,12.462044,2024-11-18 06:00:00,1012.483887,0.289761,-0.706809,3.374227,2,1,1,2,46.00,46.25,12.25,12.50


## 6. Quality Checks

Before export, a few structural checks confirm the table is internally consistent: no missing values introduced by the interpolation, the expected number of rows per station, and ERA5 values falling within physically plausible bounds for each variable.

In [9]:
print('='*55)
print('  CO-LOCATION TABLE 1 — QUALITY CHECKS')
print('='*55)

n_expected_rows = len(dpc_stations) * len(era5_times)
print(f'Expected rows (stations x hours): {n_expected_rows}')
print(f'Actual rows                     : {len(colocation_df)}')
print(f'Match                            : {"PASS" if len(colocation_df) == n_expected_rows else "FAIL"}')

era5_value_cols = [c for c in colocation_df.columns if c.startswith('era5_')]
n_nan = colocation_df[era5_value_cols].isna().sum()
print(f'\nMissing values per variable:')
print(n_nan.to_string())

plausibility = {
    'era5_msl': (950, 1060),
    'era5_u10': (-60, 60),
    'era5_v10': (-60, 60),
    'era5_i10fg': (0, 80),
}
print(f'\nPhysical plausibility check:')
for col, (lo, hi) in plausibility.items():
    if col in colocation_df.columns:
        out_of_range = ((colocation_df[col] < lo) | (colocation_df[col] > hi)).sum()
        print(f'  {col:12s}: {out_of_range} values outside [{lo}, {hi}]')

rows_per_station = colocation_df.groupby('station_id').size()
print(f'\nRows per station: min={rows_per_station.min()}, max={rows_per_station.max()} '
      f'(expected {len(era5_times)} for every station)')

  CO-LOCATION TABLE 1 — QUALITY CHECKS
Expected rows (stations x hours): 6605568
Actual rows                     : 6605568
Match                            : PASS

Missing values per variable:
era5_msl      0
era5_u10      0
era5_v10      0
era5_i10fg    0
era5_lat_0    0
era5_lat_1    0
era5_lon_0    0
era5_lon_1    0

Physical plausibility check:
  era5_msl    : 0 values outside [950, 1060]
  era5_u10    : 0 values outside [-60, 60]
  era5_v10    : 0 values outside [-60, 60]
  era5_i10fg  : 0 values outside [0, 80]

Rows per station: min=8784, max=8784 (expected 8784 for every station)


## 7. Export

In [ ]:
colocation_df.to_csv(OUTPUT_CSV, index=False)

print('='*55)
print('  EXPORT COMPLETE')
print('='*55)
print(f'  Saved -> {OUTPUT_CSV}')
print(f'  Rows  : {len(colocation_df)}')
print(f'  Cols  : {len(colocation_df.columns)}')
print(f'  Columns: {list(colocation_df.columns)}')
print()
print('  This table provides, for every DPC station and every hour of 2024,')
print('  the bilinearly interpolated ERA5 value at that station\'s exact')
print('  coordinates — independent of whether a DPC observation exists for')
print('  that timestamp. It is intended as a contextual background feature')
print('  and as a pre-fine-tuning baseline, NOT as a temporally paired')
print('  training target.')

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.